# ENet compression sweep report -- 4-class objective

Single reporting point for the current experiment: 4-class segmentation
(LAD/RCA/LCX/LM on `Dataset509_ARCADE_1x1_4c`), replacing the retired binary
(single foreground class) study on `Dataset501_ARCADE` -- see
`compression/README.md`'s top note and `compression/slurm/archive/README.md`.
One section per stage; each reads `compression/results.csv` filtered by
`stage` and renders that stage's table + plot. Re-run end-to-end after each
stage's runs land -- this notebook is how checkpoints get reported, not ad
hoc scripts.

`dice`/`cldice` are the MEAN across the 4 foreground classes
(`compression/collect_results.py`); per-class `dice_LAD`/`dice_RCA`/
`dice_LCX`/`dice_LM` (and the `cldice_*` equivalents) are also in
`results.csv` and shown in each stage's table alongside the mean.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1] if (Path.cwd().name == "notebook") else Path.cwd()
COMPRESSION_DIR = REPO_ROOT / "compression"
RESULTS_CSV = COMPRESSION_DIR / "results.csv"
COST_TABLES_DIR = COMPRESSION_DIR / "cost_tables"
RESULTS_DIR = COMPRESSION_DIR / "results"

PER_CLASS_DICE_COLS = ["dice_LAD", "dice_RCA", "dice_LCX", "dice_LM"]


def load_all() -> pd.DataFrame:
    if not RESULTS_CSV.exists():
        print(f"{RESULTS_CSV} does not exist yet -- no runs collected. Run collect_results.py first.")
        return pd.DataFrame()
    return pd.read_csv(RESULTS_CSV)


def load_stage(stage: str) -> pd.DataFrame:
    df = load_all()
    if df.empty:
        return df
    stage_df = df[df["stage"] == stage].copy()
    if stage_df.empty:
        print(f"No rows yet for stage={stage!r}.")
    return stage_df


def show_table(df: pd.DataFrame, columns: list[str] | None = None) -> None:
    if df.empty:
        return
    if columns:
        columns = [c for c in columns if c in df.columns]
    display(df[columns] if columns else df)


def plot_dice_vs(df: pd.DataFrame, x_col: str, title: str, label_col: str = "config_name") -> None:
    if df.empty or x_col not in df.columns or "dice" not in df.columns:
        return
    plotted = df.dropna(subset=[x_col, "dice"])
    if plotted.empty:
        print(f"Nothing to plot for {title} (missing {x_col} or dice values).")
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(plotted[x_col], plotted["dice"])
    for _, row in plotted.iterrows():
        ax.annotate(str(row[label_col]), (row[x_col], row["dice"]), fontsize=8,
                    textcoords="offset points", xytext=(4, 4))
    ax.set_xlabel(x_col)
    ax.set_ylabel("Dice (mean of LAD/RCA/LCX/LM)")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


## Stage 1_naive_baseline -- channel-width grid

Baseline/U2/U4/U8/U16: uniform divisors of the ENet-paper baseline
(16,64,128,64,16), decoder_type=upsample_conv, PReLU off, everything else
at ENet-native defaults. U4 is the reference every later stage's probes are
built off of.


In [ ]:
stage1 = load_stage("1_naive_baseline")
show_table(stage1, ["config_name", "f_i", "f1", "f2", "f3", "f4", "f5",
                     "params", "flops", "dice", *PER_CLASS_DICE_COLS, "cldice", "n_components"])
plot_dice_vs(stage1, "params", "Stage 1_naive_baseline: Dice vs. Params")
plot_dice_vs(stage1, "flops", "Stage 1_naive_baseline: Dice vs. FLOPs")

if not stage1.empty and stage1["dice"].notna().any():
    best = stage1.loc[stage1["dice"].idxmax()]
    print(f"Best so far: {best['config_name']} (dice={best['dice']:.4f}, params={best['params']:.0f}, flops={best['flops']:.0f})")


## Stage 2_special_ops -- single-flag ablations off U4

Each probe flips exactly one knob away from U4 (PReLU, decoder type,
dilated, asymmetric) to re-test findings from the retired binary run against
the new 4-class objective. See `compression/slurm/stage_2_special_ops_array.job`.


In [ ]:
stage2 = load_stage("2_special_ops")
show_table(stage2, ["config_name", "decoder_type", "ops_flags", "params", "flops",
                     "dice", *PER_CLASS_DICE_COLS, "cldice", "n_components"])
plot_dice_vs(stage2, "params", "Stage 2_special_ops: Dice vs. Params")


## Stage 3_transfer_original -- fine-tuned from the Dataset501 checkpoint

Warm-starts ENet_Original's exact architecture (16,64,128,64,16,
decoder_type=max_unpool, PReLU on) from its own binary-run checkpoint
instead of training from scratch -- see
`compression/slurm/stage_3_transfer_original.job`.


In [ ]:
stage3 = load_stage("3_transfer_original")
show_table(stage3, ["config_name", "decoder_type", "params", "flops",
                     "dice", *PER_CLASS_DICE_COLS, "cldice", "n_components"])


## Cost tables -- architecture-only marginal cost (no training)

Per-stage marginal params/FLOPs for +1 filter / +1 bottleneck block, off
the (16,64,128,64,16) baseline -- `compression/generate_cost_tables.py`.
Activation memory is no longer part of this (dropped this session; only
params/MACs/Dice feed the ranking below now).


In [ ]:
filter_cost_path = COST_TABLES_DIR / "filter_cost.csv"
bottleneck_cost_path = COST_TABLES_DIR / "bottleneck_cost.csv"
if filter_cost_path.exists():
    display(pd.read_csv(filter_cost_path))
else:
    print(f"{filter_cost_path} not generated yet -- run generate_cost_tables.py.")
if bottleneck_cost_path.exists():
    display(pd.read_csv(bottleneck_cost_path))
else:
    print(f"{bottleneck_cost_path} not generated yet -- run generate_cost_tables.py.")


## Stage 4_arch_probes -- architecture probes off U4

8 probes, each changing exactly one thing off U4: E1-shape channels, extra
initial features, shallow-stage dilation, separable dilated convs, merged
dilated pairs, dilation-only DSC, doubled 1x1 projections, two-block skip.
See `compression/slurm/stage_4_arch_probes_array.job`.


In [ ]:
stage4 = load_stage("4_arch_probes")
show_table(stage4, ["config_name", "f_i", "f1", "f2", "f3", "f4", "f5", "ops_flags",
                     "params", "flops", "dice", *PER_CLASS_DICE_COLS, "cldice", "n_components"])
plot_dice_vs(stage4, "params", "Stage 4_arch_probes: Dice vs. Params")


## Ranking -- cost-vs-accuracy score

`score = (macs_ratio + params_ratio - dice_ratio) / 3` (equal-weighted;
50/50 macs/params only, no Dice term, for rows without a trained Dice yet)
vs. `nnUNetTrainerENet_1_naive_baseline_Baseline` -- see
`compression/rank_results.py`. Lower = cheaper, weighted against accuracy
loss.


In [ ]:
ranking_path = RESULTS_DIR / "ranking.csv"
if ranking_path.exists():
    display(pd.read_csv(ranking_path))
else:
    print(f"{ranking_path} not generated yet -- run rank_results.py.")
